# Sesión 9 · Plantilla del proyecto final
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

> Ejecute cada celda con `Shift + Enter` y lea la explicación antes de cada paso. Todos los datos son **sintéticos** (simulados) para practicar.

**Objetivo:** estructurar su proyecto: pregunta, datos, modelo base, modelo avanzado y métrica. Complete las celdas marcadas con **TODO**.

## 1. Ficha del proyecto
Escriba en una frase cada punto. Ejemplo: *Predecir la generación solar horaria de una estación para planear el despacho del día siguiente.*

In [ ]:
ficha = {
    'Integrantes': 'TODO',
    'Pregunta de investigación': 'TODO',
    'Variable a predecir': 'TODO',
    'Decisión que apoya': 'TODO',
    'Fuente de datos (cite)': 'TODO',
    'Periodo y frecuencia': 'TODO',
    'Métrica de éxito': 'TODO (ej. MAE en kW)',
    'Línea base': 'TODO (ej. persistencia)',
}
for k, v in ficha.items(): print(f'{k}: {v}')

## 2. Cargar los datos
Opción A: suba su archivo CSV (por ejemplo de datos.gov.co o del IDEAM) y cámbielo en `RUTA`. Opción B: use los datos sintéticos del curso (por defecto).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
RUTA = None   # TODO: 'mis_datos.csv'
if RUTA:
    df = pd.read_csv(RUTA)
else:
    rng = np.random.default_rng(1)
    idx = pd.date_range('2025-01-01', periods=24*90, freq='h')
    h = idx.hour.values
    sol = np.clip(np.sin((h-6)/12*np.pi),0,None)
    irr = np.clip(1000*sol*(1-np.clip(rng.normal(0.3,0.2,len(idx)),0,0.9)),0,None)
    df = pd.DataFrame({'fecha':idx,'irr':irr,'pot':0.18*irr+rng.normal(0,2,len(idx))})
print(df.shape)
df.head()

## 3. Revisión rápida de calidad
Recuerde la sesión 4: tipos, faltantes, duplicados y atípicos.

In [ ]:
print(df.dtypes)
print('Faltantes por columna:'); print(df.isna().sum())
print('Duplicados:', df.duplicated().sum())
df.describe()

## 4. División por fecha y línea base

In [ ]:
from sklearn.metrics import mean_absolute_error
objetivo = 'pot'   # TODO: su variable a predecir
corte = int(len(df)*0.8)
y = df[objetivo].values
persist = y[corte-1:-1]
print('MAE persistencia (línea base):', round(mean_absolute_error(y[corte:], persist),2))

## 5. Modelo base y modelo avanzado
Entrene primero un modelo simple y luego uno más complejo. Complete las variables en `entradas`.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
entradas = ['irr']   # TODO: sus variables de entrada
Xtr, Xte = df[entradas].iloc[:corte], df[entradas].iloc[corte:]
ytr, yte = df[objetivo].iloc[:corte], df[objetivo].iloc[corte:]
modelos = {'Lineal (base)': LinearRegression(), 'Bosque (avanzado)': RandomForestRegressor(n_estimators=100, random_state=0)}
resultados = {}
for nombre, m in modelos.items():
    m.fit(Xtr, ytr); resultados[nombre] = mean_absolute_error(yte, m.predict(Xte))
resultados['Persistencia'] = mean_absolute_error(y[corte:], persist)
print(pd.Series(resultados).round(2).sort_values())

## 6. Gráfica de resultados
Una buena gráfica cuenta la historia: real contra predicho.

In [ ]:
mejor = modelos['Bosque (avanzado)']
n = 24*5
plt.figure(figsize=(10,3.5))
plt.plot(yte.values[:n], label='real'); plt.plot(mejor.predict(Xte)[:n], label='predicho')
plt.legend(); plt.title('Primeros 5 días del periodo de prueba'); plt.show()

## 7. Conclusiones (para completar)
- ¿El modelo supera la línea base? ¿En cuánto?
- ¿Dónde falla (horas, días nublados)?
- ¿Qué limitaciones tiene (datos, sesgos, periodo)?
- ¿Qué haría con más tiempo?
- ¿Qué decisión apoya y con qué incertidumbre?